In [119]:
import requests
url = "https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops"
html = requests.get(url)
with open("laptop_data.txt","w",encoding="utf-8") as file:
    file.write(html.text)
    print("HTML Saved")

HTML Saved


In [120]:
with open("laptop_data.txt","r",encoding="utf-8") as file:
    html = file.read()

In [121]:
from bs4 import BeautifulSoup
import pandas as pd

# Parse HTML
soup = BeautifulSoup(html, "html.parser")

# Find all product containers
containers = soup.find_all(class_="product-wrapper card-body")

# Store all laptop data
all_data = []

for container in containers:

    # =========================
    # DETAILS
    # =========================
    details = container.find(class_="card-text")

    detail_text = details.text.strip() if details else "N/A"

    # Split safely 
    parts = [p.strip() for p in detail_text.split(",")]

    # Default values
    name = "N/A"
    screen = "N/A"
    cpu = "N/A"
    ram = "N/A"
    storage = "N/A"
    os = "N/A"

    # =========================
    # NAME
    # =========================
    if len(parts) > 0:
        name = parts[0]

    # =========================
    # EXTRACT DATA USING PATTERNS
    # =========================
    for p in parts:

        # Screen size
        if '"' in p:
            screen = p

        # RAM
        elif "GB" in p and (
            "SSD" not in p
            and "HDD" not in p
            and "TB" not in p
        ):
            ram = p

        # Storage
        elif (
            "SSD" in p
            or "HDD" in p
            or "TB" in p
            or "eMMC" in p
        ):
            storage = p

        # Operating system
        elif (
            "Windows" in p
            or "Linux" in p
            or "DOS" in p
            or "Endless" in p
        ):
            os = p

        # CPU
        elif (
            "Intel" in p
            or "Core" in p
            or "Celeron" in p
            or "Pentium" in p
            or "AMD" in p
        ):
            cpu = p

    # =========================
    # PRICE
    # =========================
    price_tag = container.find("span", itemprop="price")

    amount = "N/A"

    if price_tag:

        text = price_tag.text.strip()

        if "$" in text:
            amount = text.split("$")[1].split(".")[0]

    # =========================
    # REVIEWS
    # =========================
    review_count = "N/A"

    review = container.find(itemprop="reviewCount")

    if review:
        review_count = review.text.strip()

    # =========================
    # STAR RATING
    # =========================
    star = "N/A"

    star_tag = container.find("p", attrs={"data-rating": True})

    if star_tag:
        star = star_tag.get("data-rating")

    # =========================
    # SAVE DATA
    # =========================
    data = {
        "Name": name,
        "Screen": screen,
        "CPU": cpu,
        "RAM": ram,
        "Storage": storage,
        "OS": os,
        "Price": amount,
        "Reviews": review_count,
        "Stars": star,
    }

    all_data.append(data)

# =========================
# DATAFRAME
# =========================
df = pd.DataFrame(all_data)

print(df)

# Save CSV
df.to_csv("laptops.csv", index=False)

print("\nCSV Saved Successfully")

                                           Name           Screen  \
0    Asus VivoBook X441NA-GA190 Chocolate Black              14"   
1            Prestigio SmartBook 133S Dark Grey    13.3" FHD IPS   
2                 Prestigio SmartBook 133S Gold    13.3" FHD IPS   
3                                         15.6"            15.6"   
4                             Lenovo V110-15IAP         15.6" HD   
..                                          ...              ...   
112                          Lenovo Legion Y720    15.6" FHD IPS   
113               Asus ROG Strix GL702VM-GC146T        17.3" FHD   
114               Asus ROG Strix GL702ZC-GC154T        17.3" FHD   
115               Asus ROG Strix GL702ZC-GC209T    17.3" FHD IPS   
116  Asus ROG Strix SCAR Edition GL503VM-ED115T  15.6" FHD 120Hz   

                       CPU                   RAM               Storage  \
0            Celeron N3450                   4GB             128GB SSD   
1     Celeron N3350 1.1GHz         

In [122]:
# Show all laptops costing more than $700.
df[df["Price"]>"700"]

,Name,Screen,CPU,RAM,Storage,OS,Price,Reviews,Stars
45,Asus VivoBook S14 (S406UA-BV041T) Starry Grey,"14""",Core i5-8250U,8GB,256GB SSD,Windows 10 Home,729,2,1
46,"14""","14""",Core i5 2.6GHz,500GB,N/A,N/A,739,8,4
47,Moon Silver,"15.6""",Core i7-4510U,Radeon HD R7 M265 2GB,1TB,N/A,745,12,3
48,Asus ROG STRIX GL553VD-DM256,"15.6"" FHD",Core i5-7300HQ,GeForce GTX 1050 2GB,1TB,N/A,799,7,2
49,Acer Nitro 5 AN515-51,"15.6"" FHD IPS",Core i5-7300HQ,GeForce GTX 1050 2GB,1TB,Windows 10 Home,809,0,1
50,Asus ROG STRIX GL553VD-DM256,"15.6"" FHD",Core i5-7300HQ,GeForce GTX 1050 2GB,1TB,No OS + Windows 10 Home,899,7,1
51,Lenovo ThinkPad L570,"15.6"" FHD",Core i7-7500U,8GB,256GB SSD,Windows 10 Pro,999,11,3


In [123]:
# Find laptops with 8GB RAM.
df[df["RAM"].str.contains("8GB", na=False)]

,Name,Screen,CPU,RAM,Storage,OS,Price,Reviews,Stars
8,"Acer Aspire A315-31-C33J Black 15.6""","Acer Aspire A315-31-C33J Black 15.6""",Celeron N3350,128GB,N/A,Windows 10 Home,379,0,2
16,Lenovo V110-15ISK,"15.6"" HD",Core i3-6006U,8GB,128GB SSD,Windows 10 Home,409,9,3
20,Asus EeeBook R416NA-FA014T,"14"" FHD",Pentium N4200,128GB eMMC,N/A,Windows 10 Home,433,1,1
41,"15.6""","15.6""",Core i5-4200U,8GB,1TB,Windows 8.1,581,2,1
43,Acer Aspire A515-51-5654,"15.6""",Core i5-8250U,8GB DDR4,256GB SSD,Windows 10 Home,679,9,2
45,Asus VivoBook S14 (S406UA-BV041T) Starry Grey,"14""",Core i5-8250U,8GB,256GB SSD,Windows 10 Home,729,2,1
51,Lenovo ThinkPad L570,"15.6"" FHD",Core i7-7500U,8GB,256GB SSD,Windows 10 Pro,999,11,3
53,Lenovo ThinkPad L460,"14"" FHD IPS",Core i7-6600U,8GB,256GB SSD,Windows 10 Pro,1096,14,4
57,"Apple MacBook Air 13.3""","Apple MacBook Air 13.3""",Intel HD 4000,8GB,128GB SSD,N/A,1101,4,2
58,Dell Latitude 5280,"12.5"" HD",Core i5-7300U,8GB,256GB SSD,Windows 10 Pro,1102,8,1


In [124]:
# Count total Acer laptops.
df[df["Name"].str.contains("Acer")]

,Name,Screen,CPU,RAM,Storage,OS,Price,Reviews,Stars
7,Acer Aspire 3 A315-31 Black,"15.6"" HD",Celeron N3350 1.1GHz,4GB,128GB SSD,Windows 10 Home,372,2,2
8,"Acer Aspire A315-31-C33J Black 15.6""","Acer Aspire A315-31-C33J Black 15.6""",Celeron N3350,128GB,N/A,Windows 10 Home,379,0,2
9,Acer Aspire ES1-572 Black,"15.6"" HD",Core i3-6006U,4GB,128GB SSD,Linux,379,9,4
10,Acer Aspire 3 A315-31 Black,"15.6"" HD",Pentium N4200 1.1GHz,4GB,128GB SSD,Windows 10 Home,391,10,4
11,Acer Aspire 3 A315-21,"15.6""",N/A,N/A,AMD A4-9120. 4GB. 128GB SSD,Linux,393,9,3
15,Acer Aspire 3 A315-31 Black,"15.6"" HD",Pentium N4200 1.1GHz,4GB,128GB SSD,Windows 10 Home,408,10,4
17,Acer Aspire ES1-732 Black,"17.3"" HD+",Celeron,4GB,1TB,Windows 10 Home,410,14,4
21,Acer Aspire 3 A315-51,"15.6"" HD",Core i3-6006U,4GB,1TB,Windows 10 Home,436,1,1
22,Acer Aspire ES1-572 Black,"15.6"" HD",Core i3-6006U,500GB,N/A,Windows 10 Home,436,2,1
23,Acer Extensa 15 (2540) Black,"15.6"" HD",Core i5-7200U,500GB,N/A,Linux,439,6,4


In [128]:
# Find the average laptop price.
df["Price"].mean()

np.float64(908.9230769230769)

In [27]:
# Show only SSD storage laptops.
df[df["Storage"].str.contains("SSD")]

,Name,Screen,CPU,RAM,Storage,OS,Price,Reviews,Stars
0,Asus VivoBook X441NA-GA190 Chocolate Black,"14""",Celeron N3450,4GB,128GB SSD,Endless OS,295,14,3
4,Lenovo V110-15IAP,"15.6"" HD",Celeron N3350 1.1GHz,4GB,128GB SSD,Windows 10 Home,321,5,3
6,Hewlett Packard 250 G6 Dark Ash Silver,"15.6"" HD",Celeron N3060 1.6GHz,4GB,128GB SSD,DOS,364,12,1
7,Acer Aspire 3 A315-31 Black,"15.6"" HD",Celeron N3350 1.1GHz,4GB,128GB SSD,Windows 10 Home,372,2,2
9,Acer Aspire ES1-572 Black,"15.6"" HD",Core i3-6006U,4GB,128GB SSD,Linux,379,9,4
...,...,...,...,...,...,...,...,...,...
111,Asus ASUSPRO B9440UA-GV0279R Gray,"14"" FHD",Core i7-7500U,16GB,512GB SSD,Windows 10 Pro,1381,4,1
112,Lenovo Legion Y720,"15.6"" FHD IPS",Core i7-7700HQ,RGB backlit keyboard,128GB SSD + 2TB HDD,DOS,1399,8,3
113,Asus ROG Strix GL702VM-GC146T,"17.3"" FHD",Core i7-7700HQ,GeForce GTX 1060 3GB,1TB + 128GB SSD,Windows 10 Home,1399,10,3
115,Asus ROG Strix GL702ZC-GC209T,"17.3"" FHD IPS",N/A,Radeon RX 580 4GB,256GB SSD + 1TB HDD,Windows 10 Home,1769,8,1


In [103]:
# Find the most expensive laptop.
df[df["Price"==df["Price"].max()]]

KeyError: np.False_